In [112]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import re

In [113]:
# URL of the Berlin universities page
url = 'https://edurank.org/geo/de-berlin/'

# Send a GET request to the URL
response = requests.get(url)

In [114]:
# Parse the HTML content using BeautifulSoup
parsed_content = bs(response.content, 'html.parser')

In [115]:
# Find all university blocks
univ_blocks = parsed_content.find_all('div', class_='block-cont') # Each block corresponds to a university

In [116]:
# Initialize an empty list to store university information
page_text = re.sub(r'\s+', ' ', ' '.join(str(block) for block in univ_blocks))  # Replace multiple spaces/newlines with a single space

In [ ]:
#fetching an example of text page
print(page_text[:200])  # Print the first 200 characters of the page text

In [117]:
# Function to extract ranks from the page text
def rank_extracter(page_text):
    ranks = {
        # Matches: #<rank> of <total> In Berlin
        'berlin_rank': r'#(\d+)\s+of\s+[\d,]+\s*In\s+Berlin',
        # Matches: #<rank> of <total> In Germany
        'germany_rank': r'#(\d+)\s+of\s+[\d,]+\s*In\s+Germany',
        # Matches: #<rank> of <total> In Europe
        'europe_rank': r'#(\d+)\s+of\s+[\d,]+\s*In\s+Europe',
        # Matches: #<rank> of <total> In the World
        'world_rank': r'#(\d+)\s+of\s+[\d,]+\s*In\s+the\s+World'
    }
    results = {}
    for category, pattern in ranks.items():
        matched = re.search(pattern, page_text, re.IGNORECASE)
        if matched:
            rank_value = matched.group(1)  # Extract the rank value
            print(f"Category: {category}, Matched: {matched.group(0)}")  # Debugging output
            results[category] = rank_value if rank_value else 'N/A'
        else:
            results[category] = 'N/A'

    return results

In [ ]:
# Initialize an empty list to store university information
univ_info = []

# Loop through each university block
for univ in univ_blocks:
    # Extract the university name and link
    univ_name_element = univ.find('h2', class_='h4 font-weight-bold text-center')
    if univ_name_element and univ_name_element.find('a'):
        univ_name = univ_name_element.find('a').text.strip()
        # Remove numerical prefix
        univ_name = re.sub(r'^\d+\.\s*', '', univ_name)
        univ_link = univ_name_element.find('a')['href']
    else:
        univ_name = 'N/A'
        univ_link = None

    # If a link is found, extract the ranking information
    if univ_link:
        if univ_link.startswith('https://'):
            full_univ_link = univ_link
        else:
            full_univ_link = f'https://edurank.org{univ_link}'
        try:
            univ_response = requests.get(full_univ_link)
            univ_response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
            univ_parsed_content = bs(univ_response.content, 'html.parser')
            univ_page_text = univ_parsed_content.get_text()
            rankings = extract_rankings(univ_page_text)

            # Store the data
            univ_data = {'University Name': univ_name, **rankings}  # Merge name and rankings
            univ_info.append(univ_data)

        except requests.exceptions.RequestException as e:
            print(f"Error fetching {full_univ_link}: {e}")
            univ_data = {'University Name': univ_name,
                         'berlin_rank': 'N/A',
                         'germany_rank': 'N/A',
                         'europe_rank': 'N/A',
                         'world_rank': 'N/A'}
            univ_info.append(univ_data)

    else:
        # If no link is found, store the name with N/A rankings
        univ_data = {'University Name': univ_name,
                     'berlin_rank': 'N/A',
                     'germany_rank': 'N/A',
                     'europe_rank': 'N/A',
                     'world_rank': 'N/A'}
        univ_info.append(univ_data)

# Create a DataFrame from the scraped data
df = pd.DataFrame(univ_info)

In [120]:
# Drop rows where all columns have 'N/A'
df = df[~(df == 'N/A').all(axis=1)]

# Display the updated DataFrame
display(df)

,University Name,berlin_rank,germany_rank,europe_rank,world_rank
1,Free University of Berlin,1,5,5,33
2,Humboldt University of Berlin,2,9,9,49
3,Technical University of Berlin,3,19,19,79
4,Charite - Medical University of Berlin,4,21,21,100
5,Berlin University of Applied Sciences,5,84,84,634
6,Berlin School of Economics and Law,6,91,91,730
7,Berlin Technical University of Applied Sciences,7,95,95,756
8,Berlin University of the Arts,8,98,98,785
9,Hertie School of Governance,9,99,99,790
10,Steinbeis University,10,135,135,993


In [ ]:
#fetching an example of text page
print(page_text[:100])  # Print the first 100 characters of the page text

<div class="block-cont mb-3 pb-1"> <div class="mb-3"> <div class="block-cont__offset"> <!-- a# --> <


Issues to be solved:

1) the values in the column 'world_rank', should be the values for 'europe_rank'

2) the values in the column 'europe_rank' are identic to the values of 'germany_rank'

3) the regex code doesn't fetsch the values for 'world_rank' 